# Implementing Causal Self-Attention

By the end of this notebook, you will understand the "self-attention" mechanism that lets tokens in a sequence communicate with each other. You will implement it from scratch, including the crucial "causal mask" that prevents the model from cheating by looking into the future. This will give you a foundational understanding of how models like GPT generate text one token at a time.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

# Set a seed for reproducibility
torch.manual_seed(42)

### The Core Idea: Attention as a Networking Event

Imagine a sentence is a room full of people, where each person is a word (or token). "Self-attention" is like a networking event for these words. Each word wants to understand itself better by interacting with the other words in the room.

To do this, each word plays three roles:

1.  **Query (Q):** As a "Querier," a word asks a question that expresses its own identity and what it's looking for. For example, the word "kicked" might ask, "Who is doing the kicking, and what is being kicked?"

2.  **Key (K):** As a "Key-holder," a word holds a label that describes what it is. The word "ball" might have a key that says, "I am a noun, a thing that can be kicked."

3.  **Value (V):** As a "Value-provider," a word holds the actual information or meaning it contributes. The word "ball" has the value of, well, a ball.

The process is simple: The Querier ("kicked") walks around the room and compares its question (its Query) to every other word's label (their Key). If a Key is a good match for the Query (like "ball" is for "kicked"), the Querier pays a lot of attention to that word. The strength of this match determines how much of that word's Value the Querier incorporates into its own understanding. After networking, "kicked" is no longer just an action; it's an action connected to a "ball".

**The "Causal" Rule**

Now, for a generative model like GPT, there's one critical rule at this networking event: **You can only talk to people who arrived *before* you.**

This is "causal" or "masked" attention. When the model is trying to predict the next word in a sentence, it's only allowed to see the words that came before it. The word "kicked" can pay attention to "the" and "boy", but it absolutely cannot see the word "ball" if it's trying to predict it. This mask ensures the model learns to generate text properly, one step at a time, without cheating by peeking at the answer.

We implement this by calculating attention scores between all words, but then "masking out" the scores for all future words before turning them into weights.

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    A simplified implementation of Causal Self-Attention.
    """
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_embd = n_embd
        self.n_head = n_head
        
        # A single linear layer to produce query, key, and value projections
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)
        # Output projection
        self.c_proj = nn.Linear(n_embd, n_embd)

        # The causal mask
        # We register it as a buffer so it's part of the module's state,
        # but not considered a parameter to be trained.
        # `torch.tril` creates a lower-triangular matrix.
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("bias", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        # Batch size, Sequence length, Embedding dimensionality
        B, T, C = x.size() 

        # 1. Get query, key, value from a single projection
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # 2. Reshape for multi-head attention
        # (B, T, C) -> (B, T, n_head, head_size) -> (B, n_head, T, head_size)
        head_size = C // self.n_head
        k = k.view(B, T, self.n_head, head_size).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_size).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_size).transpose(1, 2)

        # 3. Calculate attention scores ("affinities")
        # (B, nh, T, hs) @ (B, nh, hs, T) -> (B, nh, T, T)
        att_scores = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        # 4. Apply the causal mask
        # `masked_fill_` sets elements to -inf where the mask is 0
        att_scores = att_scores.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        
        # 5. Softmax to get attention weights
        att_weights = F.softmax(att_scores, dim=-1)

        # 6. Weigh the values by the attention weights
        # (B, nh, T, T) @ (B, nh, T, hs) -> (B, nh, T, hs)
        y = att_weights @ v

        # 7. Re-assemble the heads
        # (B, nh, T, hs) -> (B, T, nh, hs) -> (B, T, C)
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # 8. Final output projection
        y = self.c_proj(y)
        return y, att_weights # Return weights for visualization

### A Line-by-Line Walkthrough

Let's break down the `forward` method step-by-step.

1.  **`q, k, v = self.c_attn(x).split(...)`**: We pass the input `x` through a single `Linear` layer that triples the embedding dimension. Then we immediately split the result into three equal chunks to get our Query, Key, and Value vectors for every token. This is more efficient than using three separate linear layers.

2.  **`k.view(...).transpose(...)`**: This is the multi-head shuffle. We reshape the `(Batch, Time, Channels)` tensor to split the `Channels` dimension into `n_head` different heads. We then transpose the `Time` and `n_head` dimensions so that we can perform matrix multiplication across heads in parallel. Analogy: We're arranging our "networking event" into multiple, smaller conversation circles (`n_head`) instead of one large one.

3.  **`att_scores = (q @ k.transpose(...))`**: This is the heart of attention. We perform a matrix multiplication between every token's Query (`q`) and every other token's Key (`k`). This produces a `(T, T)` matrix of similarity scores. A high value at `att_scores[i, j]` means token `i` finds token `j` very relevant. The scaling by `1/sqrt(head_size)` is a standard trick to keep the scores from becoming too large, which helps stabilize training.

4.  **`att_scores.masked_fill(...)`**: This is where we enforce causality. `self.bias` is our lower-triangular matrix of 1s and 0s. Where the mask is `0` (the upper-right triangle, representing future tokens), we replace the attention score with negative infinity.

5.  **`att_weights = F.softmax(...)`**: Softmax turns the raw scores into a probability distribution. The scores we set to negative infinity will become `0.0` after softmax. This means a token will assign exactly zero attention to any future token.

6.  **`y = att_weights @ v`**: We now have our attention weights. We use them to create a weighted sum of the Value vectors. Each token's output representation becomes a blend of other tokens' values, based on how much attention it paid to them.

7.  **`y.transpose(...).contiguous().view(...)`**: We reverse the multi-head shuffle from step 2. We concatenate the outputs from all the heads back into a single vector for each token, restoring the original `(B, T, C)` shape.

8.  **`y = self.c_proj(y)`**: A final `Linear` layer projects the combined head outputs. This allows the model to learn how to best mix the information gathered from the different attention heads.

In [ ]:
# --- Demonstration ---

# Hyperparameters for our model
batch_size = 1
block_size = 8  # Max sequence length
n_embd = 32     # Embedding dimension
n_head = 4      # Number of attention heads

# Create the CausalSelfAttention module
attention_layer = CausalSelfAttention(n_embd=n_embd, n_head=n_head, block_size=block_size)

# Create a dummy input tensor
# (batch_size, sequence_length, embedding_dim)
# We use a sequence length of 5, which is less than block_size
dummy_input = torch.randn(batch_size, 5, n_embd) 

# Pass the input through the layer
output, att_weights = attention_layer(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", att_weights.shape)

# The output shape should be the same as the input shape
assert output.shape == dummy_input.shape

# The attention weights shape is (B, n_head, T, T)
assert att_weights.shape == (batch_size, n_head, 5, 5)

print("\nSuccess! The output shape matches the input shape.")

### Going Deeper: One Big Projection vs. Three Small Ones

A non-obvious design decision in the nanoGPT implementation is the use of a single `Linear` layer, `c_attn`, to generate the Query, Key, and Value vectors all at once.

```python
# In nanoGPT
self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)

# A more obvious, but less efficient, alternative:
# self.query_proj = nn.Linear(config.n_embd, config.n_embd)
# self.key_proj = nn.Linear(config.n_embd, config.n_embd)
# self.value_proj = nn.Linear(config.n_embd, config.n_embd)
# q = self.query_proj(x)
# k = self.key_proj(x)
# v = self.value_proj(x)
```

Why do it this way?

**Performance.** Modern GPUs are highly optimized for executing a few large matrix multiplications rather than many small ones. The overhead of launching three separate computation "kernels" for Q, K, and V is higher than launching one kernel for a single, larger matrix multiplication. By projecting to `3 * n_embd` and then splitting, we perform the core computation in one efficient step. This is a common optimization in high-performance deep learning code.

This pattern of fusing operations is a recurring theme in performant model implementations.

In [ ]:
# --- Visualization of the Causal Mask ---

# We can directly inspect the attention weights we got from the demonstration
# Let's look at the weights from the first head in the batch
attention_matrix = att_weights[0, 0, :, :].detach().numpy()

# The sequence length was 5, so we have a 5x5 matrix
print("Attention Matrix (Head 1):\n", np.round(attention_matrix, 2))

fig, ax = plt.subplots()
cax = ax.imshow(attention_matrix, cmap='viridis')
ax.set_title("Causal Attention Weights (One Head)")
ax.set_xlabel("Key Token Position (Information Source)")
ax.set_ylabel("Query Token Position (Information Receiver)")
ax.set_xticks(np.arange(attention_matrix.shape[1]))
ax.set_yticks(np.arange(attention_matrix.shape[0]))
ax.xaxis.set_ticks_position('top') 
ax.xaxis.set_label_position('top')
fig.colorbar(cax, label="Attention Weight")
plt.show()

The visualization clearly shows the effect of our causal mask. Notice the strict lower-triangular pattern:

-   **Token 0** (the first word) can only attend to itself.
-   **Token 2** can attend to itself (token 2) and the tokens before it (0 and 1), but all weights for future tokens (3 and 4) are zero.
-   The upper-right triangle of the matrix is all dark blue (zero), confirming that no token was able to "peek" into the future.

This is the core mechanism that allows a Transformer to be an **auto-regressive** model—one that generates a sequence by predicting the next element based only on the previous ones.

### Summary & What's Next

**What we built:** We implemented a `CausalSelfAttention` module from scratch, the engine that allows tokens in a sequence to selectively share information.

**Key Takeaways:**
-   Self-attention operates on Queries, Keys, and Values, allowing each token to create a new representation of itself by taking a weighted average of all other tokens' values.
-   The "causal" part is a simple but powerful mask that forces the model to only attend to past and present tokens, making it suitable for generation tasks.
-   Multi-head attention allows the model to focus on different types of token relationships in parallel, making it more powerful than a single attention mechanism.

**What's Next:**
This attention module is only one piece of the puzzle. In the next notebook, **"Constructing the Transformer Block,"** we will combine our `CausalSelfAttention` layer with a standard feed-forward neural network (MLP) to build a complete `Block`. This block is the fundamental, repeating unit that makes up the GPT architecture.